# 04.P · Mini-project: a leak-free churn pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/project/project.ipynb)

Read the [brief](README.md) first. Fill in every `TODO`. The first three cells are done for you: they load the data and define how a customer table is built for any cut-off date.

When you finish, compare with [solution.ipynb](solution.ipynb).

## 0. Load and clean (done)

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])
print(f"{len(sales):,} clean transactions from {sales['InvoiceDate'].min():%Y-%m-%d} to {sales['InvoiceDate'].max():%Y-%m-%d}")

## 1. Customer table for a cut-off date (done)

Read this function carefully: it is where leakage would hide.

In [ ]:
def build_table(cutoff):
    """One row per customer active before `cutoff`.
    Features use only orders BEFORE the cut-off; churned = no order in the 3 months AFTER it."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=3))]
    t = past.groupby("Customer ID").agg(
        orders=("Invoice", "nunique"), spend=("Revenue", "sum"), products=("StockCode", "nunique"),
        first=("InvoiceDate", "min"), last=("InvoiceDate", "max"), country=("Country", lambda s: s.mode()[0]))
    t["tenure_days"] = (cutoff - t["first"]).dt.days
    t["days_since_last"] = (cutoff - t["last"]).dt.days
    t["avg_basket"] = t["spend"] / t["orders"]
    recent = past[past["InvoiceDate"] >= cutoff - pd.Timedelta(days=90)].groupby("Customer ID")["Invoice"].nunique()
    t["orders_last_90d"] = recent.reindex(t.index).fillna(0)
    t["churned"] = (~t.index.isin(future["Customer ID"].unique())).astype(int)
    return t.drop(columns=["first", "last"])

## 2. Time-based split (TODO)

Build the **training** table with cut-off `2011-06-01` and the **test** table with cut-off `2011-09-01`. Define `num_cols` (seven numeric columns) and `cat_cols` (`country`). Print size and churn rate of each.

In [ ]:
# TODO

## 3. A quick look (TODO)

On the **training** table only, show churn rate by recency band (0–30, 31–90, 91–180, 181+ days). One sentence on what you see.

In [ ]:
# TODO

## 4. Preprocessing (TODO)

Write a `ColumnTransformer`: numeric → median imputer → `log1p` → scaler; `country` → most-frequent imputer → one-hot with rare countries grouped.

In [ ]:
# TODO

## 5. Compare three models with cross-validation (TODO)

Put the preprocessing and each model into a `Pipeline`: logistic regression, KNN (k = 101) and a decision tree (max_depth = 5). Use 5-fold **stratified** CV on the training table only, scoring ROC AUC. Report mean ± std in a small table.

In [ ]:
# TODO

## 6. Choose and tune (TODO)

Pick a model and justify the choice in one or two sentences (score *and* anything else that matters). Tune one hyperparameter by CV on the training table.

In [ ]:
# TODO

## 7. The one final test (TODO)

Fit the chosen pipeline on the whole training table. Score it **once** on the test table: AUC, accuracy at threshold 0.5, confusion matrix, and the churn rate among the 500 customers with the highest predicted probability.

In [ ]:
# TODO

## 8. Leakage audit (TODO)

Fill in the table: for each risk, say how your notebook avoids it (or does not).

| Risk | How this notebook handles it |
|---|---|
| Features using data after the cut-off | |
| Test outcomes used for training | |
| Preprocessing fitted on test or held-out rows | |
| Model chosen by looking at the test score | |
| A feature that is really the target in disguise | |

## 9. Conclusion (TODO)

Five or six sentences for the retention manager: how good is the model, how sure are you, what would you do with it on 1 September, and what is the next step to improve it?